# Continuous FIQA · retrieval-conditioned threshold calibration

00의 완료 FIQA/condition score를 입력으로 Global-safe → FIQA 2/5-bin → Continuous FIQA → +ADC margin → +Gallery PQ distortion을 비교합니다. Runner-up score 단독 추가도 보조 대조군으로 제공합니다.

**먼저 quality-only 연속형을 실행하고, 이후 retrieval feature 비교를 실행하세요.** 모든 설정은 첫 번째 코드 셀에 있습니다. Saliency의 1차 목적(압축과 인식 근거·검색 행동의 관계 분석)은 00 및 기존 분석에서 유지하며, 2차 추가정보 검증은 향후 02에서 수행합니다.


In [1]:
# 0. 사용자 설정 — 이 셀에서 변경 후 Kernel Restart → Run All
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (p / 'research').is_dir() and (p / '.git').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 루트에서 실행하세요.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SURVFACE_RUN_CANDIDATES = {
    'arcface': PROJECT_ROOT / 'runs/survface_20260902/20260902-R001-61915edf_step4_survface_arcface-7972a704552df378345f',
    'adaface': PROJECT_ROOT / 'runs/survface_20260830/20260830-R001-ec6e5d4a_step4_survface_adaface-4df25b75e065b0b9ed43',
    'magface': PROJECT_ROOT / 'runs/survface_20260831/20260831-R001-6695386d_step4_survface_magface-6931178ad2025e1b3799',
    'edgeface': PROJECT_ROOT / 'runs/survface_20260901/20260901-R001-56c2f3ed_step4_survface_edgeface-a348c305af33c223b337',
}
SOURCE_MODEL = 'edgeface'
FIQA_VARIANT = 'L'
COMPRESSION_PROFILE = 'pq_512_m128_b8'
SEARCH_MODE = 'pq_adc_exhaustive'
METRIC_CONTRACT = 'genuine-score-topk-v2'
RESULT_ROOT = PROJECT_ROOT / 'results/calibration'

# 단일 split 먼저 확인. 같은 seed와 hyperparameter를 test 결과에 맞춰 선택하지 않습니다.
PARTITION_SEED = 8972
TARGET_FPIRS = (.01, .05, .10, .20, .30)
SAFETY_FRACTION = .30
SHRINKAGE_STRENGTH = 200.
MINIMUM_GROUP_NON_MATED = 100
KNOT_QUANTILES = (1/3, 2/3)
SMOOTHING = .01          # fit score 표준편차 단위의 smoothed pinball 폭
RIDGE = .001
MAX_ITERATIONS = 2000
MARGIN_SLOPE_CAP = .95  # raw margin 계수 범위 [0, .95]; 임의 test 조절 금지
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 8972

# A. 기본 False: FIQA 단독 Continuous만 수행. True: margin/distortion/runner-up 추가.
USE_RETRIEVAL_FEATURES = True
BUILD_RETRIEVAL_FEATURES = True  # calibration+test ADC top-2 검색 및 shard 저장; 중단 후 재개
FEATURE_ARTIFACT_DIR = None      # 이미 생성한 fiqa-retrieval-UID 디렉터리의 명시적 Path
FEATURE_BATCH_SIZE = 8192
FEATURE_SCORE_TOLERANCE = 1e-6
# B. 연속형 비교 (저장 없이 메모리 내 계산만 수행할 수도 있습니다.)
RUN_CONTINUOUS_CALIBRATION = True
WRITE_CONTINUOUS_RESULTS = True
# C. 동일 cohort의 전체 fit/safety 분할 안정성 (고비용; 단일 split 다음에 수행)
RUN_SPLIT_STABILITY = True
WRITE_SPLIT_RESULTS = True
SPLIT_SEEDS = (*range(19), 8972)

if SOURCE_MODEL not in SURVFACE_RUN_CANDIDATES or FIQA_VARIANT not in ('S', 'L'):
    raise ValueError('명시된 source model/FIQA variant를 선택하세요.')
if WRITE_CONTINUOUS_RESULTS and not RUN_CONTINUOUS_CALIBRATION:
    raise ValueError('WRITE_CONTINUOUS_RESULTS requires RUN_CONTINUOUS_CALIBRATION')
if WRITE_SPLIT_RESULTS and not RUN_SPLIT_STABILITY:
    raise ValueError('WRITE_SPLIT_RESULTS requires RUN_SPLIT_STABILITY')
if BUILD_RETRIEVAL_FEATURES and not USE_RETRIEVAL_FEATURES:
    raise ValueError('BUILD_RETRIEVAL_FEATURES requires USE_RETRIEVAL_FEATURES')

if USE_RETRIEVAL_FEATURES:
    if BUILD_RETRIEVAL_FEATURES and FEATURE_ARTIFACT_DIR is not None:
        raise ValueError('feature build와 명시적 기존 경로 로드 중 하나만 선택하세요.')
    if not BUILD_RETRIEVAL_FEATURES and FEATURE_ARTIFACT_DIR is None:
        raise ValueError('retrieval 비교에는 feature build 또는 완료 feature 경로가 필요합니다.')
if RUN_SPLIT_STABILITY and (len(SPLIT_SEEDS) < 2 or len(set(SPLIT_SEEDS)) != len(SPLIT_SEEDS)):
    raise ValueError('분할 안정성에는 서로 다른 seed 두 개 이상이 필요합니다.')


## 실행 방법 — 첫 코드 셀만 변경

현재 기본 RUN/BUILD/WRITE는 False입니다. **그대로 Run All 하면 입력 확인만 하고 성능 실험은 하지 않습니다.** 저장된 출력은 과거 실행 기록이므로 변경 후에는 Kernel Restart → Run All 하세요.

| 단계 | 첫 셀 설정 | 실행 범위 |
|---|---|---|
| A. Continuous FIQA만 | RUN_CONTINUOUS_CALIBRATION=True | Global-safe, 2/5-bin, Continuous |
| B. Margin/PQ distortion 추가 | 위 설정 + USE_RETRIEVAL_FEATURES=True, BUILD_RETRIEVAL_FEATURES=True | frozen ADC feature 추출 + 7개 방법 비교 |
| C. 완료 feature 재사용 | USE_RETRIEVAL_FEATURES=True, BUILD_RETRIEVAL_FEATURES=False, FEATURE_ARTIFACT_DIR=명시적 완료 경로 | ADC 재검색 없이 비교 |
| D. 분할 안정성 | RUN_SPLIT_STABILITY=True | 같은 방법들을 사전 지정 20개 seed로 반복 |

A/B/C 결과 저장은 WRITE_CONTINUOUS_RESULTS=True, D 저장은 WRITE_SPLIT_RESULTS=True로 켭니다. C에서도 단일 split 비교에는 RUN_CONTINUOUS_CALIBRATION=True가 필요합니다. B의 feature build 자체는 재개용 shard를 저장합니다. 이 노트북은 FR/FIQA 추론을 반복하지 않으며 00에서 준비한 완료 입력이 필요합니다.


## 1. 방법과 해석 계약

- 연속형: FIQA에 fit 분위수 두 개의 hinge knot를 둔 **smoothed quantile regression**. Test가 아닌 calibration fit의 non-mated score로 적합합니다. 품질 범위 밖은 fit 경계로 clip합니다.
- Margin/runner-up/distortion은 선형 항으로 추가합니다. 모든 스케일과 knot는 fit에서만 정합니다. Margin과 runner-up은 별도 방법입니다.
- Safety: `r = non_mated_score - fitted_threshold(features)`의 held-out 분위수로 `max(0, residual_threshold)`를 더합니다. 분포 전이에서의 formal FPIR guarantee는 아닙니다.
- ADC margin은 서로 다른 gallery identity template의 `s1-s2`입니다. Margin raw 기울기를 1 미만으로 제한하지만, gallery 후보가 바뀌는 상황 전체의 단조성을 보장하지는 않습니다.
- PQ distortion은 frozen codec 입력 공간에서 `||g - decode(PQ(g))||_2`. 복원 후 재정규화하지 않으며 query 방향의 정확한 score error는 아닙니다.
- FPIR은 non-mated top-1 score, TPIR20은 **genuine identity score의 threshold 통과 AND rank ≤ 20**으로 평가합니다.
- TPIR CI는 인물 단위, FPIR CI는 unknown identity 부재로 query 단위입니다. 고정 threshold CI와 seed별 기술통계이며 test 기반 방법/seed 선택을 하지 않습니다.


In [2]:
import json
import pandas as pd
from IPython.display import display, Markdown
from research.fiqa import CRFIQA_VARIANTS, load_fiqa_score_artifact
from research.experiments.fiqa_threshold_calibration import load_condition_score_artifact
from research.experiments.fiqa_retrieval_features import build_retrieval_features, load_retrieval_features
from research.experiments.fiqa_continuous_calibration import run_continuous_calibration, write_continuous_calibration
from research.runtime.hashing import sha256_file


In [3]:
planned_method_count = 7 if USE_RETRIEVAL_FEATURES else 4
display(pd.DataFrame([
    {'stage': 'single split', 'enabled': RUN_CONTINUOUS_CALIBRATION,
     'method_target_evaluations': planned_method_count * len(TARGET_FPIRS) if RUN_CONTINUOUS_CALIBRATION else 0,
     'write': WRITE_CONTINUOUS_RESULTS},
    {'stage': 'split stability', 'enabled': RUN_SPLIT_STABILITY,
     'method_target_evaluations': planned_method_count * len(TARGET_FPIRS) * len(SPLIT_SEEDS) if RUN_SPLIT_STABILITY else 0,
     'write': WRITE_SPLIT_RESULTS},
]))
if not RUN_CONTINUOUS_CALIBRATION and not RUN_SPLIT_STABILITY:
    display(Markdown('**성능 비교 비활성:** 현재 설정은 입력/feature 준비만 수행합니다.'))


,stage,enabled,method_target_evaluations,write
0,single split,True,35,True
1,split stability,True,700,True


In [4]:
# 2. 00에서 생성한 명시적 완료 입력을 hash 검증하여 로드
source_run_dir = SURVFACE_RUN_CANDIDATES[SOURCE_MODEL]
if not (source_run_dir / 'COMPLETED').is_file():
    raise FileNotFoundError(f'완료 run이 없습니다: {source_run_dir}')
source_manifest = json.loads((source_run_dir / 'run_manifest.json').read_text(encoding='utf8'))
if source_manifest.get('status') != 'completed':
    raise ValueError('source run이 completed 상태가 아닙니다.')
source_run_id = source_manifest['run_id']
condition_dir = RESULT_ROOT / 'condition_scores' / source_run_id / f'{COMPRESSION_PROFILE}__{SEARCH_MODE}' / METRIC_CONTRACT
fiqa_dir = RESULT_ROOT / 'fiqa_scores/survface' / CRFIQA_VARIANTS[FIQA_VARIANT].model_uid
if not (condition_dir / 'manifest.json').is_file():
    raise FileNotFoundError(f'00에서 먼저 완료 condition을 생성하세요: {condition_dir}')
condition = load_condition_score_artifact(condition_dir)
fiqa = load_fiqa_score_artifact(fiqa_dir)
for key, expected in {'source_run_id': source_run_id, 'model_uid': source_manifest['config']['model_uid'],
                      'compression_profile': COMPRESSION_PROFILE, 'search_mode': SEARCH_MODE,
                      'metric_contract': METRIC_CONTRACT,
                      'source_run_manifest_sha256': sha256_file(source_run_dir / 'run_manifest.json')}.items():
    if condition.manifest.get(key) != expected:
        raise ValueError(f'condition lineage 불일치: {key}')
display(pd.DataFrame([{'run_id': source_run_id, 'fiqa': FIQA_VARIANT,
                      'calibration_rows': len(condition.calibration), 'test_rows': len(condition.test),
                      'score_space': condition.manifest['score_space']}]))


,run_id,fiqa,calibration_rows,test_rows,score_space
0,20260901-R001-56c2f3ed,L,160408,182159,negative_squared_l2_adc


## 3. 선택 단계 — ADC top-2와 PQ distortion feature

Quality-only에서는 이 단계를 건너뜁니다. Retrieval 비교 시 첫 실행은 `USE_RETRIEVAL_FEATURES=True`, `BUILD_RETRIEVAL_FEATURES=True`로 설정합니다.

각 batch의 ADC top-1 score가 00의 저장 score와 일치해야만 shard를 기록합니다. 완료 shard는 다음 실행에서 hash 검증 후 재사용합니다. Faiss CPU ADC 검색이며 FIQA inference·SDC 실행은 없습니다. 이후 feature 디렉터리를 `FEATURE_ARTIFACT_DIR`에 지정하면 replay 없이 읽습니다. 전체 완료 전에는 비교에 사용할 수 없습니다.


In [5]:
retrieval = None
if USE_RETRIEVAL_FEATURES:
    if FEATURE_ARTIFACT_DIR is not None:
        if BUILD_RETRIEVAL_FEATURES:
            raise ValueError('기존 feature 경로 로드와 새 build 중 하나만 선택하세요.')
        retrieval = load_retrieval_features(FEATURE_ARTIFACT_DIR, condition)
    elif BUILD_RETRIEVAL_FEATURES:
        retrieval = build_retrieval_features(
            source_run_dir, condition, RESULT_ROOT / 'fiqa_retrieval_features' / source_run_id,
            batch_size=FEATURE_BATCH_SIZE, score_tolerance=FEATURE_SCORE_TOLERANCE,
            progress=lambda event: print(event, flush=True),
        )
    else:
        raise ValueError('Retrieval 비교에는 FEATURE_ARTIFACT_DIR 또는 BUILD_RETRIEVAL_FEATURES=True가 필요합니다.')
    display(Markdown(f'완료 feature: `{retrieval["directory"]}`'))
else:
    display(Markdown('Quality-only: 기존 FIQA/condition 점수만 사용합니다.'))


{'split': 'calibration', 'queries_done': 8192, 'total': 160408}
{'split': 'calibration', 'queries_done': 16384, 'total': 160408}
{'split': 'calibration', 'queries_done': 24576, 'total': 160408}
{'split': 'calibration', 'queries_done': 32768, 'total': 160408}
{'split': 'calibration', 'queries_done': 40960, 'total': 160408}
{'split': 'calibration', 'queries_done': 49152, 'total': 160408}
{'split': 'calibration', 'queries_done': 57344, 'total': 160408}
{'split': 'calibration', 'queries_done': 65536, 'total': 160408}
{'split': 'calibration', 'queries_done': 73728, 'total': 160408}
{'split': 'calibration', 'queries_done': 81920, 'total': 160408}
{'split': 'calibration', 'queries_done': 90112, 'total': 160408}
{'split': 'calibration', 'queries_done': 98304, 'total': 160408}
{'split': 'calibration', 'queries_done': 106496, 'total': 160408}
{'split': 'calibration', 'queries_done': 114688, 'total': 160408}
{'split': 'calibration', 'queries_done': 122880, 'total': 160408}
{'split': 'calibration'

완료 feature: `C:\ronbun\results\calibration\fiqa_retrieval_features\20260901-R001-56c2f3ed\fiqa-retrieval-77406544899535c41d714d0b`

## 4. 단일 split 비교

상단 `RUN_CONTINUOUS_CALIBRATION=True`로 계산하고, `WRITE_CONTINUOUS_RESULTS=True`로 저장합니다. 모델 계수·전처리·safety offset은 models.csv의 model_json에 함께 저장합니다.

기본: Global-safe / FIQA 2-bin / FIQA 5-bin / Continuous FIQA. Retrieval 활성화 시 +Margin / +Margin+Distortion / +Runner-up을 추가합니다. 먼저 실제 FPIR 목표 충족 여부와 CI를 보고 TPIR20의 변화량을 읽으세요.


In [6]:
methods = ('continuous_fiqa',)
if USE_RETRIEVAL_FEATURES:
    methods += ('continuous_fiqa_margin', 'continuous_fiqa_margin_distortion', 'continuous_fiqa_runnerup')
common_settings = dict(
    retrieval=retrieval, methods=methods, target_fpirs=TARGET_FPIRS,
    safety_fraction=SAFETY_FRACTION, shrinkage_strength=SHRINKAGE_STRENGTH,
    minimum_group_non_mated=MINIMUM_GROUP_NON_MATED,
    knot_quantiles=KNOT_QUANTILES, smoothing=SMOOTHING, ridge=RIDGE,
    max_iterations=MAX_ITERATIONS, margin_slope_cap=MARGIN_SLOPE_CAP,
    resamples=BOOTSTRAP_RESAMPLES, bootstrap_seed=BOOTSTRAP_SEED,
    progress=lambda event: print(event, flush=True),
)
comparison = None
comparison_path = None
if RUN_CONTINUOUS_CALIBRATION:
    comparison = run_continuous_calibration(condition, fiqa, partition_seeds=(PARTITION_SEED,), **common_settings)
    if WRITE_CONTINUOUS_RESULTS:
        comparison_path = write_continuous_calibration(
            RESULT_ROOT / 'fiqa_continuous' / source_run_id, comparison)
    display(comparison['method_summary'][[
        'target_fpir', 'method', 'realized_fpir', 'fpir_wilson95_low', 'fpir_wilson95_high',
        'tpir_at_rank_k', 'tpir_cluster95_low', 'tpir_cluster95_high',
        'target_met_on_test', 'target_met_by_wilson_upper',
    ]])
    display(comparison['paired_comparisons'])
    display(comparison['models'][['target_fpir', 'method', 'fit_seconds', 'evaluation_seconds', 'model_uid']])
else:
    display(Markdown('단일 split 대기: 상단 RUN_CONTINUOUS_CALIBRATION 설정'))


{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.01, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.01, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.01, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.01, 'method': 'continuous_fiqa_runnerup'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.05, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.05, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.05, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.05, 'method': 'continuous_fiqa_runnerup'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.1, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.1, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 8972, 'target_fpir': 0.1, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 's

,target_fpir,method,realized_fpir,fpir_wilson95_low,fpir_wilson95_high,tpir_at_rank_k,tpir_cluster95_low,tpir_cluster95_high,target_met_on_test,target_met_by_wilson_upper
0,0.01,global_safe,0.011410,0.010829,0.012022,0.000017,0.000000,0.000052,False,False
1,0.01,fiqa_2bin,0.010810,0.010245,0.011407,0.000050,0.000000,0.000134,False,False
2,0.01,fiqa_5bin,0.010441,0.009885,0.011027,0.000182,0.000064,0.000344,False,False
3,0.01,continuous_fiqa,0.010630,0.010069,0.011221,0.000794,0.000438,0.001280,False,False
4,0.01,continuous_fiqa_margin,0.010630,0.010069,0.011221,0.000794,0.000438,0.001280,False,False
5,0.01,continuous_fiqa_margin_distortion,0.014178,0.013529,0.014858,0.002102,0.001053,0.003456,False,False
6,0.01,continuous_fiqa_runnerup,0.010161,0.009613,0.010740,0.008573,0.005918,0.011544,False,False
7,0.05,global_safe,0.051497,0.050269,0.052752,0.001341,0.000804,0.001969,False,False
8,0.05,fiqa_2bin,0.051078,0.049855,0.052329,0.001638,0.001007,0.002428,False,False
9,0.05,fiqa_5bin,0.051669,0.050440,0.052927,0.002929,0.002112,0.003847,False,False


,partition_seed,target_fpir,reference_method,candidate_method,metric,reference_successes,candidate_successes,both_successes,total,candidate_minus_reference,paired_bootstrap95_low,paired_bootstrap95_high,resampling_unit,resamples,bootstrap_seed
0,8972,0.01,global_safe,fiqa_2bin,fpir,1389,1316,1145,121736,-0.000600,-0.000928,-0.000279,query,2000,8972
1,8972,0.01,global_safe,fiqa_2bin,tpir_at_rank_k,1,3,1,60423,0.000033,0.000000,0.000084,mated_identity_cluster,2000,8972
2,8972,0.01,global_safe,fiqa_5bin,fpir,1389,1271,1059,121736,-0.000969,-0.001347,-0.000608,query,2000,8972
3,8972,0.01,global_safe,fiqa_5bin,tpir_at_rank_k,1,11,1,60423,0.000165,0.000050,0.000321,mated_identity_cluster,2000,8972
4,8972,0.01,global_safe,continuous_fiqa,fpir,1389,1294,1046,121736,-0.000780,-0.001183,-0.000386,query,2000,8972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,8972,0.30,continuous_fiqa,continuous_fiqa_margin_distortion,tpir_at_rank_k,2081,2024,1376,60423,-0.000943,-0.004553,0.003095,mated_identity_cluster,2000,8972
126,8972,0.30,continuous_fiqa,continuous_fiqa_runnerup,fpir,35451,36208,12948,121736,0.006218,0.002883,0.009899,query,2000,8972
127,8972,0.30,continuous_fiqa,continuous_fiqa_runnerup,tpir_at_rank_k,2081,1742,684,60423,-0.005610,-0.013273,0.001484,mated_identity_cluster,2000,8972
128,8972,0.30,continuous_fiqa_margin,continuous_fiqa_margin_distortion,fpir,35451,39495,25505,121736,0.033219,0.030722,0.035824,query,2000,8972


,target_fpir,method,fit_seconds,evaluation_seconds,model_uid
0,0.01,global_safe,NaN,0.330181,conditional-threshold-cc8be883f60a6731b5174bc8
1,0.01,fiqa_2bin,NaN,0.333014,conditional-threshold-53e9bcf9a6eb493a115de180
2,0.01,fiqa_5bin,NaN,0.329402,conditional-threshold-2ff1a753d86a59162fcb7606
3,0.01,continuous_fiqa,0.725634,0.341214,continuous-threshold-42984da82faa6cec41512a91
4,0.01,continuous_fiqa_margin,0.665483,0.334667,continuous-threshold-56660fb33cd6cca2cead1de9
5,0.01,continuous_fiqa_margin_distortion,0.954688,0.356863,continuous-threshold-fd3b8ab4482984591f7cca83
6,0.01,continuous_fiqa_runnerup,0.768022,0.330336,continuous-threshold-588cc772518935457ff7479b
7,0.05,global_safe,NaN,0.317826,conditional-threshold-e75a172f6a9cc0ec758ab244
8,0.05,fiqa_2bin,NaN,0.327955,conditional-threshold-e205a8457b6207aca970843c
9,0.05,fiqa_5bin,NaN,0.324602,conditional-threshold-b6142a3f72695e75a46ac074


## 4.1 단계별 추가 효과

Continuous와 2/5-bin의 비교, Continuous → +Margin, +Margin → +Margin+Distortion을 같은 seed와 query에서 직접 비교합니다. Runner-up은 별도 보조 대조군입니다.

ablation_summary.csv의 delta는 **후속 방법 − 이전 방법**이며 0~1 단위입니다(×100 = %p). FPIR 변화와 TPIR20 변화를 함께 읽으세요. TPIR CI가 양수여도 candidate_misses_target이면 목표 FPIR에서의 개선이라고 결론 내리지 않습니다. both_point_targets_met도 동일한 실제 FPIR을 강제한 비교나 이론적 보장을 의미하지 않습니다.

FPIR은 query, TPIR20은 mated identity cluster CI입니다. 두 paired CI 모두 첫 셀의 BOOTSTRAP_RESAMPLES/BOOTSTRAP_SEED를 따릅니다. multiple comparison 및 calibration 재적합 불확실성을 포함하지 않는 탐색적 비교입니다.


In [7]:
if comparison is not None:
    display(comparison['ablation_summary'][[
        'target_fpir', 'stage', 'reference_method', 'candidate_method',
        'reference_realized_fpir', 'candidate_realized_fpir',
        'delta_fpir', 'delta_fpir_ci_low', 'delta_fpir_ci_high',
        'delta_tpir_at_rank_k', 'delta_tpir_at_rank_k_ci_low', 'delta_tpir_at_rank_k_ci_high',
        'tpir_ci_direction', 'operating_point_status',
    ]])
else:
    display(Markdown('단계별 비교 대기: 단일 split 실행 후 표시합니다.'))


,target_fpir,stage,reference_method,candidate_method,reference_realized_fpir,candidate_realized_fpir,delta_fpir,delta_fpir_ci_low,delta_fpir_ci_high,delta_tpir_at_rank_k,delta_tpir_at_rank_k_ci_low,delta_tpir_at_rank_k_ci_high,tpir_ci_direction,operating_point_status
0,0.01,continuous_vs_2bin,fiqa_2bin,continuous_fiqa,0.010810,0.010630,-0.000181,-0.000542,0.000140,0.000745,0.000385,0.001229,increase,candidate_misses_target
1,0.01,continuous_vs_5bin,fiqa_5bin,continuous_fiqa,0.010441,0.010630,0.000189,-0.000115,0.000485,0.000612,0.000245,0.001085,increase,candidate_misses_target
2,0.01,add_margin,continuous_fiqa,continuous_fiqa_margin,0.010630,0.010630,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,includes_zero,candidate_misses_target
3,0.01,add_gallery_distortion,continuous_fiqa_margin,continuous_fiqa_margin_distortion,0.010630,0.014178,0.003549,0.002834,0.004263,0.001307,0.000446,0.002497,increase,candidate_misses_target
4,0.01,runnerup_control,continuous_fiqa,continuous_fiqa_runnerup,0.010630,0.010161,-0.000468,-0.001249,0.000321,0.007778,0.005206,0.010518,increase,candidate_misses_target
5,0.05,continuous_vs_2bin,fiqa_2bin,continuous_fiqa,0.051078,0.051530,0.000452,-0.000230,0.001093,0.001969,0.001182,0.002879,increase,candidate_misses_target
6,0.05,continuous_vs_5bin,fiqa_5bin,continuous_fiqa,0.051669,0.051530,-0.000140,-0.000854,0.000542,0.000679,0.000093,0.001290,increase,candidate_misses_target
7,0.05,add_margin,continuous_fiqa,continuous_fiqa_margin,0.051530,0.051530,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,includes_zero,candidate_misses_target
8,0.05,add_gallery_distortion,continuous_fiqa_margin,continuous_fiqa_margin_distortion,0.051530,0.063112,0.011582,0.010251,0.012971,0.003161,0.001285,0.005709,increase,candidate_misses_target
9,0.05,runnerup_control,continuous_fiqa,continuous_fiqa_runnerup,0.051530,0.050955,-0.000575,-0.002218,0.001191,0.011022,0.007631,0.014599,increase,candidate_misses_target


## 5. 선택 단계 — 분할 안정성

동일 gallery/test와 codec을 고정하고 calibration 내부 fit/safety만 다시 나눕니다. `SPLIT_SEEDS` 전체에서 같은 설정을 적용하며, 최적 seed나 모델을 자동 선택하지 않습니다. 최소·중앙·최대는 기술통계로 95% CI가 아닙니다. 단일 split 결과와 별도 UID로 저장합니다.


In [8]:
stability = None
stability_path = None
if RUN_SPLIT_STABILITY:
    if len(SPLIT_SEEDS) < 2:
        raise ValueError('분할 안정성에는 두 개 이상의 서로 다른 seed가 필요합니다.')
    stability = run_continuous_calibration(condition, fiqa, partition_seeds=SPLIT_SEEDS, **common_settings)
    if WRITE_SPLIT_RESULTS:
        stability_path = write_continuous_calibration(
            RESULT_ROOT / 'fiqa_continuous_split_stability' / source_run_id, stability)
    display(stability['split_summary'])
else:
    display(Markdown('분할 안정성 대기: 상단 RUN_SPLIT_STABILITY 설정'))


{'stage': 'fit', 'seed': 0, 'target_fpir': 0.01, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.01, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.01, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.01, 'method': 'continuous_fiqa_runnerup'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.05, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.05, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.05, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.05, 'method': 'continuous_fiqa_runnerup'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.1, 'method': 'continuous_fiqa'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.1, 'method': 'continuous_fiqa_margin'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.1, 'method': 'continuous_fiqa_margin_distortion'}
{'stage': 'fit', 'seed': 0, 'target_fpir': 0.1, 'met

,method,target_fpir,split_count,target_met_split_count,fpir_min,fpir_median,fpir_max,tpir_min,tpir_median,tpir_max
0,continuous_fiqa,0.01,20,1,0.009915,0.010654,0.011098,0.000728,0.000778,0.000877
1,continuous_fiqa,0.05,20,3,0.048400,0.050725,0.052055,0.003409,0.003591,0.003757
2,continuous_fiqa,0.10,20,20,0.090877,0.098102,0.099371,0.007994,0.008838,0.008970
3,continuous_fiqa,0.20,20,20,0.185311,0.194511,0.198076,0.020721,0.021565,0.022061
4,continuous_fiqa,0.30,20,20,0.279416,0.291270,0.295336,0.033050,0.034532,0.035003
5,continuous_fiqa_margin,0.01,20,1,0.009915,0.010654,0.011098,0.000728,0.000778,0.000877
6,continuous_fiqa_margin,0.05,20,3,0.048400,0.050725,0.052055,0.003409,0.003591,0.003757
7,continuous_fiqa_margin,0.10,20,20,0.090877,0.098102,0.099371,0.007994,0.008838,0.008970
8,continuous_fiqa_margin,0.20,20,20,0.185311,0.194511,0.198076,0.020721,0.021565,0.022061
9,continuous_fiqa_margin,0.30,20,20,0.279416,0.291270,0.295336,0.033050,0.034532,0.035003


## 6. 결과와 다음 단계

새 실험은 `results/calibration/fiqa_continuous` 및 `fiqa_continuous_split_stability` 아래 content UID로 저장됩니다. 기존 완료 결과는 덮어쓰지 않고 재사용 시 manifest/CSV hash를 검증합니다. 실행 소스가 바뀌면 새로운 UID를 사용합니다.

공통 보고서는 기존 pinned evidence를 계속 읽습니다. 새 결과를 공통 보고에 반영하려면 별도의 명시적 연결과 검증이 필요합니다. 02의 saliency 추가정보 검증은 아직 수행하지 않습니다. 이 노트북에서 high/low/random saliency 가중치로 threshold를 조절하지 않습니다.

산출물: method_summary.csv(방법별 FPIR/TPIR/CI), paired_comparisons.csv(모든 직접 비교), ablation_summary.csv(단계별 추가 효과), models.csv(적합 계수/안전 보정), split_summary.csv(분할 기술통계). 새 continuous 결과는 schema_version=2이며 metric_contract는 genuine-score-topk-v2입니다. 다른 artifact의 schema_version과 혼동하지 마세요.


In [9]:
display(pd.DataFrame([
    {'stage': 'retrieval features', 'status': 'completed' if retrieval is not None else 'not_requested',
     'artifact': str(retrieval['directory']) if retrieval is not None else None},
    {'stage': 'single split', 'status': 'computed' if comparison is not None else 'not_run',
     'artifact': str(comparison_path) if comparison_path is not None else None},
    {'stage': 'split stability', 'status': 'computed' if stability is not None else 'not_run',
     'artifact': str(stability_path) if stability_path is not None else None},
]))


,stage,status,artifact
0,retrieval features,completed,C:\ronbun\results\calibration\fiqa_retrieval_f...
1,single split,computed,C:\ronbun\results\calibration\fiqa_continuous\...
2,split stability,computed,C:\ronbun\results\calibration\fiqa_continuous_...
